In [1]:
from pymongo import MongoClient

# 1. Conectar al contenedor de Mongo
client = MongoClient('mongodb://mongodb:27017/')

# 2. Crear (o usar) la base de datos y la colección
db = client['TiendaBigData'] 
coleccion = db['despegardb'] # Ideal el nombre del grupo ejem: 'G_Ecommerce_scraping'

print("Conexión exitosa a MongoDB")

Conexión exitosa a MongoDB


In [ ]:
# --- PASO 0: LIMPIEZA TOTAL Y REPARACIÓN ---
import os
import time
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Cierra procesos viejos que hayan quedado abiertos
os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
os.system("rm -rf /tmp/.com.google.Chrome.*")
os.system("rm -rf /tmp/.org.chromium.Chromium.*")
print("🧹 Limpieza de procesos y temporales completada.")

# --- VARIABLES GENERALES ---
NOMBRE_GRUPO = "Vannessa"
datos_finales = []   # Se define fuera del try para que siempre exista
driver = None        # Se define fuera del try para poder cerrarlo con seguridad

# --- PASO 1: CONFIGURACIÓN DEL NAVEGADOR ---
options = Options()
options.binary_location = "/usr/bin/google-chrome"  # Ruta del binario de Chrome

# Argumentos de estabilidad para Docker
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--disable-software-rasterizer")
options.add_argument("--window-size=1920,1080")
options.add_argument("--remote-debugging-port=9222")

# User-Agent común
options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/120.0.0.0 Safari/537.36"
)

try:
    # Inicia el navegador
    driver = webdriver.Chrome(options=options)
    print(" Navegador iniciado correctamente.")

    # --- PASO 2: NAVEGACIÓN Y EXTRACCIÓN ---
    limite_paginas = 3
    driver.get("https://www.amazon.es/s?k=laptops")

    for nivel_pagina in range(limite_paginas):
        print(f"--- Procesando Página {nivel_pagina + 1} ---")

        # Espera fija para que la página termine de cargar
        time.sleep(10)

        # Espera explícita a que existan resultados
        WebDriverWait(driver, 20).until(
            EC.presence_of_all_elements_located(
                (By.CSS_SELECTOR, "div[data-component-type='s-search-result']")
            )
        )

        bloques = driver.find_elements(
            By.CSS_SELECTOR, "div[data-component-type='s-search-result']"
        )

        for bloque in bloques:
            try:
                nombre = bloque.find_element(By.TAG_NAME, "h2").text
                precio = bloque.find_element(By.CSS_SELECTOR, ".a-price-whole").text

                datos_finales.append({
                    "identificador": nombre,
                    "valor": precio,
                    "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                    "grupo": NOMBRE_GRUPO
                })
            except:
                # Si el bloque no tiene nombre o precio, se salta
                continue

        # Intenta avanzar a la siguiente página
        try:
            btn_sig = driver.find_element(By.CLASS_NAME, "s-pagination-next")
            driver.execute_script("arguments[0].click();", btn_sig)
            time.sleep(5)
        except:
            print("No se encontró el botón siguiente o ya es la última página.")
            break

    print(f" Extracción terminada: {len(datos_finales)} productos.")

except Exception as e:
    print(f" Error en Selenium: {e}")

finally:
    # Cierra el navegador solo si logró abrirse
    if driver is not None:
        try:
            driver.quit()
        except:
            pass

# --- PASO 3: GUARDAR EN MONGODB ---
try:
    client = MongoClient("mongodb", 27017, serverSelectionTimeoutMS=5000)
    db = client["TiendaBigData"]
    coleccion = db["AmazonLaptops"]

    if datos_finales:
        for d in datos_finales:
            # Limpia el valor antes de convertirlo
            v_limpio = str(d["valor"]).replace(".", "").replace(",", "").strip()
            d["valor"] = float(v_limpio) if v_limpio.isdigit() else 0.0

        coleccion.insert_many(datos_finales)
        print(" Datos cargados en MongoDB correctamente.")
    else:
        print(" No hay datos para guardar.")

except Exception as e:
    print(f" Error en MongoDB: {e}")
    

In [2]:
# --- PASO 0: LIMPIEZA TOTAL Y REPARACIÓN ---
import os
import time
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Cierra procesos viejos que hayan quedado abiertos
os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
os.system("rm -rf /tmp/.com.google.Chrome.*")
os.system("rm -rf /tmp/.org.chromium.Chromium.*")
print("🧹 Limpieza de procesos y temporales completada.")

# --- VARIABLES GENERALES ---
NOMBRE_GRUPO = "Vannessa"
datos_finales = []
driver = None

# --- PASO 1: CONFIGURACIÓN DEL NAVEGADOR ---
options = Options()
options.binary_location = "/usr/bin/google-chrome"

# Argumentos de estabilidad para Docker
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--disable-software-rasterizer")
options.add_argument("--window-size=1920,1080")
options.add_argument("--remote-debugging-port=9222")

# User-Agent común
options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/120.0.0.0 Safari/537.36"
)

try:
    # Inicia el navegador
    driver = webdriver.Chrome(options=options)
    print("🚗 Navegador iniciado correctamente.")

    # --- PASO 2: NAVEGACIÓN A DESPEGAR CARS ---
    # URL que me mandaste
    driver.get("https://www.us.despegar.com/cars/")
    
    # Esperar carga inicial
    time.sleep(8)
    
    # Buscar campo de ubicación (ciudad) y llenarlo
    print("📍 Configurando búsqueda...")
    try:
        # Selector común para campo de ubicación en Despegar
        ubicacion = WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "input[placeholder*='city'], input[placeholder*='location'], input[placeholder*='ubicación'], input[type='search']"))
        )
        ubicacion.clear()
        ubicacion.send_keys("Cancun")  # Cambia por la ciudad que quieras
        time.sleep(2)
        
        # Hacer clic en el primer resultado de la lista desplegable
        primer_resultado = driver.find_element(By.CSS_SELECTOR, "[data-testid='autocomplete-option'], .autocomplete-option, .suggestion-item")
        primer_resultado.click()
        time.sleep(1)
    except Exception as e:
        print(f"⚠️ No se pudo llenar ubicación automáticamente: {e}")
        print("📌 Continuando con la página por defecto...")
    
    # Buscar y hacer clic en el botón de buscar
    try:
        boton_buscar = driver.find_element(By.CSS_SELECTOR, "button[type='submit'], [data-testid='search-button'], .search-button")
        boton_buscar.click()
        print("🔍 Búsqueda iniciada...")
        time.sleep(8)
    except:
        print("⚠️ No se encontró botón de búsqueda, continuando con la página actual...")
    
    # --- EXTRACCIÓN DE DATOS ---
    limite_registros = 300
    pagina_actual = 1
    
    while len(datos_finales) < limite_registros:
        print(f"--- Procesando Página {pagina_actual} ---")
        
        # Esperar a que carguen los resultados
        time.sleep(8)
        
        # Esperar explícita a que existan resultados de autos
        try:
            WebDriverWait(driver, 20).until(
                EC.presence_of_all_elements_located(
                    (By.CSS_SELECTOR, "[data-testid='car-card'], .product-card, .car-item, [class*='car'], [class*='vehicle']")
                )
            )
        except:
            print("⚠️ Timeout esperando resultados")
            break
        
        # Buscar todos los autos en la página
        autos = driver.find_elements(By.CSS_SELECTOR, "[data-testid='car-card'], .product-card, .car-item, [class*='car'], [class*='vehicle']")
        
        if not autos:
            print("❌ No se encontraron autos en esta página")
            break
        
        print(f"📊 Encontrados {len(autos)} autos en página {pagina_actual}")
        
        for auto in autos:
            if len(datos_finales) >= limite_registros:
                break
            
            try:
                # Extraer NOMBRE (probando múltiples selectores)
                nombre = "No disponible"
                selectores_nombre = ["h3", ".title", ".name", "[class*='title']", "[class*='name']", ".car-name"]
                for selector in selectores_nombre:
                    try:
                        elemento = auto.find_element(By.CSS_SELECTOR, selector)
                        if elemento.text.strip():
                            nombre = elemento.text.strip()
                            break
                    except:
                        continue
                
                # Extraer PRECIO (probando múltiples selectores)
                precio_texto = "0"
                selectores_precio = [".price", ".amount", ".total-price", "[class*='price']", "[class*='amount']", ".daily-price"]
                for selector in selectores_precio:
                    try:
                        elemento = auto.find_element(By.CSS_SELECTOR, selector)
                        if elemento.text.strip():
                            precio_texto = elemento.text.strip()
                            break
                    except:
                        continue
                
                # Limpiar precio (solo números)
                import re
                precio_limpio = re.sub(r'[^\d.,]', '', precio_texto)
                precio_limpio = precio_limpio.replace(',', '.')
                
                # Intentar convertir a float
                try:
                    precio_valor = float(precio_limpio) if precio_limpio else 0.0
                except:
                    precio_valor = 0.0
                
                datos_finales.append({
                    "identificador": nombre,
                    "valor": precio_valor,
                    "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                    "grupo": NOMBRE_GRUPO,
                    "pagina": pagina_actual,
                    "precio_original": precio_texto
                })
                
                print(f"  [{len(datos_finales)}] {nombre[:50]} → ${precio_valor}")
                
            except Exception as e:
                print(f"  ⚠️ Error extrayendo auto: {e}")
                continue
        
        # Intentar avanzar a la siguiente página
        try:
            selectores_siguiente = ["[aria-label='Next']", ".next-page", ".pagination-next", "a:contains('Siguiente')", "button:contains('Next')"]
            btn_sig = None
            
            for selector in selectores_siguiente:
                try:
                    btn_sig = driver.find_element(By.CSS_SELECTOR, selector)
                    if btn_sig:
                        break
                except:
                    continue
            
            if btn_sig and btn_sig.is_enabled():
                driver.execute_script("arguments[0].click();", btn_sig)
                print("➡️ Avanzando a siguiente página...")
                pagina_actual += 1
                time.sleep(5)
            else:
                print("📌 No hay más páginas disponibles.")
                break
                
        except Exception as e:
            print(f"⚠️ No se pudo avanzar: {e}")
            break
    
    print(f"\n✅ Extracción terminada: {len(datos_finales)} autos capturados.")

except Exception as e:
    print(f"❌ Error en Selenium: {e}")

finally:
    # Cierra el navegador
    if driver is not None:
        try:
            driver.quit()
        except:
            pass

# --- PASO 3: GUARDAR EN MONGODB ---
try:
    # Cambia "mongodb" por "localhost" si estás localmente
    client = MongoClient("mongodb", 27017, serverSelectionTimeoutMS=5000)
    db = client["TiendaBigData"]
    coleccion = db["DespegarCars"]  # Nombre de colección específica para autos

    if datos_finales:
        coleccion.insert_many(datos_finales)
        print(f"✅ {len(datos_finales)} autos cargados en MongoDB correctamente.")
        print(f"📊 Rango de precios: ${min(d['valor'] for d in datos_finales)} - ${max(d['valor'] for d in datos_finales)}")
    else:
        print("⚠️ No hay datos para guardar.")

except Exception as e:
    print(f"❌ Error en MongoDB: {e}")

🧹 Limpieza de procesos y temporales completada.
🚗 Navegador iniciado correctamente.
📍 Configurando búsqueda...
⚠️ No se pudo llenar ubicación automáticamente: Message: 
Stacktrace:
#0 0x55ef3f830afa <unknown>
#1 0x55ef3f232265 <unknown>
#2 0x55ef3f284f76 <unknown>
#3 0x55ef3f2851b1 <unknown>
#4 0x55ef3f2d07d4 <unknown>
#5 0x55ef3f2cd969 <unknown>
#6 0x55ef3f2785cf <unknown>
#7 0x55ef3f279391 <unknown>
#8 0x55ef3f7f5fdb <unknown>
#9 0x55ef3f7f8f9d <unknown>
#10 0x55ef3f7e2798 <unknown>
#11 0x55ef3f7f9b30 <unknown>
#12 0x55ef3f7c9210 <unknown>
#13 0x55ef3f81dd48 <unknown>
#14 0x55ef3f81df18 <unknown>
#15 0x55ef3f82f56e <unknown>
#16 0x787fccc70ac3 <unknown>

📌 Continuando con la página por defecto...
⚠️ No se encontró botón de búsqueda, continuando con la página actual...
--- Procesando Página 1 ---
⚠️ Timeout esperando resultados

✅ Extracción terminada: 0 autos capturados.
⚠️ No hay datos para guardar.
